# State Estimation: EKF, UKF, and Particle Filter

Implement and compare three state estimation algorithms for fixed-wing UAV navigation.

**State vector (13D):** `[px, py, pz, u, v, w, qw, qx, qy, qz, omega_x, omega_y, omega_z]`

**Control vector (4D):** `[throttle, aileron, elevator, rudder]` *(Note: aileron set to 0 for data compatibility)*

**Measurements:**
- **100 Hz (IMU + Pitot):** `[omega_x, omega_y, omega_z, airspeed]` (4D)
- **10 Hz (GPS):** `[px, py, pz, u, v, w]` (6D)

## 1. Setup and Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm, cholesky
from scipy.stats import multivariate_normal
import pandas as pd
import time

# Load CasADi dynamics model (FAST!)
print("Loading CasADi dynamics...")
%run dynamics_casadi_cub.ipynb

# Store references to original CasADi functions
f_func_casadi = f_func  # Store original CasADi function
F_func_casadi = F_func  # Store original CasADi function

# Create numpy-friendly wrapper functions (hide CasADi conversions)
def f_func_numpy(x, u):
    """
    Dynamics function with numpy interface.
    
    Args:
        x: numpy array (13,) - state vector
        u: numpy array (4,) - control vector
        
    Returns:
        f: numpy array (13,) - state derivative
    """
    # Convert numpy -> CasADi DM -> evaluate -> convert back to numpy
    import casadi as ca
    x_dm = ca.DM(x.flatten())
    u_dm = ca.DM(u.flatten())
    result = f_func_casadi(x_dm, u_dm)  # Use original CasADi function
    return np.array(result).flatten()

def F_func_numpy(x, u):
    """
    Jacobian function with numpy interface.
    
    Args:
        x: numpy array (13,) - state vector
        u: numpy array (4,) - control vector
        
    Returns:
        F: numpy array (13, 13) - state Jacobian matrix
    """
    # Convert numpy -> CasADi DM -> evaluate -> convert back to numpy
    import casadi as ca
    x_dm = ca.DM(x.flatten())
    u_dm = ca.DM(u.flatten())
    result = F_func_casadi(x_dm, u_dm)  # Use original CasADi function
    return np.array(result)

# Override the CasADi functions with numpy-friendly versions
f_func = f_func_numpy
F_func = F_func_numpy

# Define dimensions for compatibility
n_states = 13
n_controls = 4

print(" CasADi dynamics loaded successfully!")
print(f" Functions available: f_func, F_func (numpy interface)")
print(f" State dimension: {n_states}")
print(f" Control dimension: {n_controls}")
print(" Ready for state estimation!")

Loading CasADi dynamics...
Creating CasADi dynamics with smooth approximations...
HH Sport Cub S2 parameters loaded:
  Mass: 0.057 kg
  Wingspan: 0.617 m
  Wing area: 0.05553 m²
  Max thrust: 0.56 N
Symbolic variables defined
State: [px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r] (13D)
Control: [throttle, aileron, elevator, rudder] (4D)
Control processing and airspeed calculation done (SMOOTH)
Aerodynamic coefficients calculated (with smooth stall model)
Rotation matrices and force transformations complete
Forces calculated
Moments calculated (4-channel control)
Dynamics equations assembled
Creating CasADi functions...
Testing CasADi functions...
Dynamics evaluation successful: shape (13, 1)
Jacobian evaluation successful: shape (13, 13)

Sample dynamics values (first 5 states):
  Position derivatives: [1.50000000e+01 3.55271368e-15 1.00000000e+00]
  Velocity derivatives: [ -8.6732182   -0.53942037 -28.49355385]

HH Sport Cub S2 CasADi dynamics ready
 CasADi dynamics loaded successfully

## 2. Configuration Parameters

In [2]:
# Simulation settings
dt_simulation = 0.01  # Simulation time step (s) - 100 Hz
trajectory_file_100hz = 'data/simulation_data-100hz.csv'  # High-rate IMU data
trajectory_file_10hz = 'data/simulation_data-10hz.csv'    # Low-rate GPS data

# Initial covariance (moderate uncertainty)
P0 = np.diag([
    100, 100, 100,           # Position variance (m²)
    5, 5, 5,                 # Velocity variance (m/s)²
    1.5, 1.5, 1.5, 1.5,     # Quaternion variance
    0.4, 0.4, 0.4            # Angular rate variance (rad/s)²
])

# Process noise covariance - INCREASED for stiff dynamics
# The Cub aircraft has very low inertia, causing high angular accelerations
# We need more process noise to account for modeling uncertainty
Q_state = np.diag([
    0.01, 0.01, 0.01,        # Position process noise (m²/s)
    0.1, 0.1, 0.1,           # Velocity process noise (m/s)²/s
    0.001, 0.001, 0.001, 0.001,  # Quaternion process noise
    0.5, 0.5, 0.5            # Angular rate process noise (rad/s)²/s - INCREASED!
])

# Measurement noise covariances
# High-rate measurements (100 Hz): IMU gyro + Pitot airspeed
R_100hz = np.diag([
    0.0025, 0.0025, 0.0025,  # Gyro (rad/s)²: 0.05 rad/s std
    2.25                     # Pitot airspeed (m/s)²: 1.5 m/s std
])

# Low-rate measurements (10 Hz): GPS position + Doppler velocity
R_10hz = np.diag([
    4.0, 4.0, 9.0,           # GPS position (m²): 2m, 2m, 3m std
    1.0, 1.0, 1.0            # Doppler velocity (m/s)²: 1 m/s std each
])

# Filter-specific parameters
ukf_alpha = 1e-3             # UKF spread parameter
ukf_beta = 2.0               # UKF distribution parameter (Gaussian optimal)
ukf_kappa = 0.0              # UKF secondary scaling
pf_num_particles = 1000      # Particle filter particle count

print("EKF Configuration (tuned for small aircraft with low inertia):")
print(f"  Using sub-step integration for numerical stability")
print(f"  Increased process noise on angular rates: {Q_state[10,10]:.2f} (rad/s)²/s")

EKF Configuration (tuned for small aircraft with low inertia):
  Using sub-step integration for numerical stability
  Increased process noise on angular rates: 0.50 (rad/s)²/s


## 3. Dynamics and Measurement Models

Define the dynamics propagation and measurement functions needed for the filters.

**Measurement rates:**
- **100 Hz (high-rate)**: IMU gyro (omega_x, omega_y, omega_z) + Pitot airspeed
- **10 Hz (low-rate)**: GPS position (px, py, pz) + GPS Doppler velocity (u, v, w)

In [3]:
def propagate_dynamics(x, u, dt):
    """
    Propagate state through dynamics using RK4 integration.
    
    Args:
        x: Current state (13D) [px, py, pz, u, v, w, qw, qx, qy, qz, omega_x, omega_y, omega_z]
        u: Control input (4D) [throttle, aileron, elevator, rudder]
        dt: Time step
        
    Returns:
        x_next: Next state (13D)
    """
    # For stiff dynamics (small aircraft with low inertia), use smaller sub-steps
    # This prevents numerical explosion in angular rates
    n_substeps = 5  # Use 5 substeps for better numerical stability
    dt_sub = dt / n_substeps
    
    x_current = x.copy()
    
    for _ in range(n_substeps):
    # RK4 integration - f_func now handles all conversions automatically
        k1 = f_func(x_current, u)
        k2 = f_func(x_current + dt_sub/2 * k1, u)
        k3 = f_func(x_current + dt_sub/2 * k2, u)
        k4 = f_func(x_current + dt_sub * k3, u)
    
        x_current = x_current + (dt_sub / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
        # Normalize quaternion after each substep to prevent drift
        quat_idx = slice(6, 10)
        x_current[quat_idx] = x_current[quat_idx] / np.linalg.norm(x_current[quat_idx])
    
    return x_current

In [4]:
def measurement_model_100hz(x):
    """
    High-rate measurement model (100 Hz): IMU gyro + Pitot airspeed.
    
    Args:
        x: State vector (13D)
        
    Returns:
        z: Measurement vector (4D) [omega_x, omega_y, omega_z, airspeed]
    """
    omega = x[10:13]  # Angular rates
    u, v, w = x[3], x[4], x[5]  # Body frame velocities
    airspeed = np.sqrt(u**2 + v**2 + w**2)
    
    return np.concatenate([omega, [airspeed]])


def measurement_model_10hz(x):
    """
    Low-rate measurement model (10 Hz): GPS position + Doppler velocity.
    
    Args:
        x: State vector (13D)
        
    Returns:
        z: Measurement vector (6D) [px, py, pz, u, v, w]
    """
    position = x[0:3]  # World frame position
    velocity = x[3:6]  # Body frame velocity
    
    return np.concatenate([position, velocity])

In [5]:
def measurement_jacobian_100hz(x):
    """
    Jacobian of high-rate measurement model (100 Hz).
    
    Args:
        x: State vector (13D)
        
    Returns:
        H: Measurement Jacobian (4x13)
    """
    H = np.zeros((4, 13))
    
    # Angular rates are direct measurements: omega = x[10:13]
    H[0:3, 10:13] = np.eye(3)
    
    # Airspeed measurement: h = sqrt(u^2 + v^2 + w^2)
    u, v, w = x[3], x[4], x[5]
    airspeed = np.sqrt(u**2 + v**2 + w**2)
    
    # Partial derivatives: dh/du, dh/dv, dh/dw
    if airspeed > 1e-6:  # Avoid division by zero
        H[3, 3] = u / airspeed  # dh/du
        H[3, 4] = v / airspeed  # dh/dv
        H[3, 5] = w / airspeed  # dh/dw
    else:
        H[3, 3:6] = 0.0  # Airspeed near zero, gradient undefined
    
    return H


def measurement_jacobian_10hz(x):
    """
    Jacobian of low-rate measurement model (10 Hz).
    
    Args:
        x: State vector (13D)
        
    Returns:
        H: Measurement Jacobian (6x13)
    """
    H = np.zeros((6, 13))
    
    # Position measurements: direct from state
    H[0:3, 0:3] = np.eye(3)
    
    # Velocity measurements: direct from state
    H[3:6, 3:6] = np.eye(3)
    
    return H

## 4. Load and Process Trajectory Data

Load true trajectory from CSV and generate noisy measurements.

In [6]:
# Load high-rate data (100 Hz) - contains all states and controls
data_100hz = pd.read_csv(trajectory_file_100hz)
time_100hz = data_100hz['time'].values

# CSV column order: [px, py, pz, omega_x, omega_y, omega_z, u, v, w, qw, qx, qy, qz]
# State vector order: [px, py, pz, u, v, w, qw, qx, qy, qz, omega_x, omega_y, omega_z]
# Need to reorder: position(1-3), velocity(7-9), quaternion(10-13), angular_rates(4-6)

states_100hz = np.zeros((len(data_100hz), 13))
states_100hz[:, 0:3] = data_100hz.iloc[:, 1:4].values    # position (px, py, pz)
states_100hz[:, 3:6] = data_100hz.iloc[:, 7:10].values   # velocity (u, v, w)
states_100hz[:, 6:10] = data_100hz.iloc[:, 10:14].values # quaternion (qw, qx, qy, qz)
states_100hz[:, 10:13] = data_100hz.iloc[:, 4:7].values  # angular rates (omega_x, omega_y, omega_z)

# ============================================================================
# CONTROL MAPPING CONFIGURATION
# ============================================================================
# Define which CSV control columns map to which 4-channel indices
# Format: [throttle_idx, aileron_idx, elevator_idx, rudder_idx]
# Use -1 to set a control to zero (not available in CSV)
# 
# Example mappings:
# Night Vapor (3-channel): [0, -1, 1, 2]  -> throttle=CSV[0], aileron=0, elevator=CSV[1], rudder=CSV[2]
# Cub (4-channel):         [0, 1, 2, 3]   -> direct mapping
# Throttle+Elevator only:  [0, -1, 1, -1] -> only throttle and elevator, others zero

# CONTROL_MAPPING = [0, -1, 1, 2]  # Default: Night Vapor format (3-channel)
CONTROL_MAPPING = [0, 1, 2, -1]  # Cub format (4-channel)

# ============================================================================

# Load controls with flexible mapping
controls_100hz = np.zeros((len(data_100hz), 4))  # Always 4D: [throttle, aileron, elevator, rudder]
controls_from_csv = data_100hz.iloc[:, 14:].values  # Get all control columns from CSV

print(f"Control Mapping Configuration:")
print(f"  CSV has {controls_from_csv.shape[1]} control columns")
print(f"  Mapping to 4-channel format: {CONTROL_MAPPING}")

# Apply mapping
for target_idx, source_idx in enumerate(CONTROL_MAPPING):
    if source_idx >= 0 and source_idx < controls_from_csv.shape[1]:
        controls_100hz[:, target_idx] = controls_from_csv[:, source_idx]
        print(f"    Control[{target_idx}] = CSV column {source_idx}")
    else:
        controls_100hz[:, target_idx] = 0.0
        print(f"    Control[{target_idx}] = 0 (not in CSV)")

control_names = ['throttle', 'aileron', 'elevator', 'rudder']
print(f"  Final 4-channel: {control_names}")

# Load low-rate data (10 Hz) - same trajectory, downsampled
data_10hz = pd.read_csv(trajectory_file_10hz)
time_10hz = data_10hz['time'].values

states_10hz = np.zeros((len(data_10hz), 13))
states_10hz[:, 0:3] = data_10hz.iloc[:, 1:4].values      # position
states_10hz[:, 3:6] = data_10hz.iloc[:, 7:10].values     # velocity
states_10hz[:, 6:10] = data_10hz.iloc[:, 10:14].values   # quaternion
states_10hz[:, 10:13] = data_10hz.iloc[:, 4:7].values    # angular rates

# Normalize quaternions (columns 6-10 in state vector)
for k in range(len(states_100hz)):
    quat_norm = np.linalg.norm(states_100hz[k, 6:10])
    states_100hz[k, 6:10] = states_100hz[k, 6:10] / quat_norm

for k in range(len(states_10hz)):
    quat_norm = np.linalg.norm(states_10hz[k, 6:10])
    states_10hz[k, 6:10] = states_10hz[k, 6:10] / quat_norm

print(f"\nState vector verification (first sample):")
print(f"  Position: {states_100hz[0, 0:3]}")
print(f"  Velocity: {states_100hz[0, 3:6]}")
print(f"  Quaternion: {states_100hz[0, 6:10]} (norm={np.linalg.norm(states_100hz[0, 6:10]):.6f})")
print(f"  Angular rates: {states_100hz[0, 10:13]}")
print(f"\nControl vector verification (first sample):")
print(f"  Controls: {controls_100hz[0]}")
for i, name in enumerate(control_names):
    print(f"    {name}: {controls_100hz[0, i]:.4f}")

# Convert Unix timestamps to relative time (start from 0)
time_100hz = time_100hz - time_100hz[0]
time_10hz = time_10hz - time_10hz[0]

# Compute actual time differences for each step (variable dt)
dt_array = np.diff(time_100hz)
dt_array = np.append(dt_array, dt_array[-1])  # Extend to match length

# Use high-rate data as the base for estimation (100 Hz loop)
N_steps = len(time_100hz)
dt_mean = np.mean(dt_array)  # For reporting only

print(f"\nLoaded trajectory data:")
print(f"  High-rate (100 Hz): {N_steps} steps, dt_mean = {dt_mean:.4f} s")
print(f"  Low-rate (10 Hz): {len(time_10hz)} steps")
print(f"  Duration: {time_100hz[-1]:.2f} seconds")

# Create measurement availability flags
# GPS updates happen at 10 Hz (every 10th sample at 100 Hz)
gps_decimation = 10  # 100 Hz / 10 Hz = 10
has_gps_measurement = np.zeros(N_steps, dtype=bool)
has_gps_measurement[::gps_decimation] = True  # GPS available every 10 samples

# IMU measurements available at every time step (100 Hz)
has_imu_measurement = np.ones(N_steps, dtype=bool)

Control Mapping Configuration:
  CSV has 3 control columns
  Mapping to 4-channel format: [0, 1, 2, -1]
    Control[0] = CSV column 0
    Control[1] = CSV column 1
    Control[2] = CSV column 2
    Control[3] = 0 (not in CSV)
  Final 4-channel: ['throttle', 'aileron', 'elevator', 'rudder']

State vector verification (first sample):
  Position: [-14.02024609 -33.8160663    4.80178282]
  Velocity: [ 8.50700783 -0.2250618  -0.03820336]
  Quaternion: [-0.99844085 -0.04807856  0.02315503  0.01637572] (norm=1.000000)
  Angular rates: [-0.26238933  0.02084843 -0.03067188]

Control vector verification (first sample):
  Controls: [-0.15301199 -0.05234858  0.48644668  0.        ]
    throttle: -0.1530
    aileron: -0.0523
    elevator: 0.4864
    rudder: 0.0000

Loaded trajectory data:
  High-rate (100 Hz): 7522 steps, dt_mean = 0.0100 s
  Low-rate (10 Hz): 752 steps
  Duration: 75.21 seconds


In [7]:
# Generate noisy measurements
np.random.seed(42)  # For reproducibility

# Preallocate measurement arrays
measurements_100hz = np.zeros((N_steps, 4))  # IMU gyro + airspeed
measurements_10hz = np.zeros((N_steps, 6))   # GPS position + velocity (only valid at GPS times)

for k in range(N_steps):
    # High-rate measurements (100 Hz): Always available
    z_100hz_clean = measurement_model_100hz(states_100hz[k])
    measurements_100hz[k] = z_100hz_clean + np.random.multivariate_normal(np.zeros(4), R_100hz)
    
    # Low-rate measurements (10 Hz): Only at GPS update times
    if has_gps_measurement[k]:
        z_10hz_clean = measurement_model_10hz(states_100hz[k])
        measurements_10hz[k] = z_10hz_clean + np.random.multivariate_normal(np.zeros(6), R_10hz)

print("Generated noisy measurements:")
print(f"  IMU (100 Hz): {N_steps} measurements")
print(f"  GPS (10 Hz): {np.sum(has_gps_measurement)} measurements")

Generated noisy measurements:
  IMU (100 Hz): 7522 measurements
  GPS (10 Hz): 753 measurements


In [8]:
# Plot true trajectory and noisy measurements
# YOUR CODE HERE
# Create subplots showing:
# - 3D position (true vs measurements)
# - Velocity components (true vs measurements)
# - Angular rates (true vs measurements)
# - Airspeed (true vs measurements)

## 5. Results Storage

Initialize dictionary structure to store results from all filters.

In [9]:
results = {
    'ekf': {
        'estimates': np.zeros((N_steps, 13)),
        'covariances': [],
        'rmse': {},
        'compute_time': 0.0
    },
    'ukf': {
        'estimates': np.zeros((N_steps, 13)),
        'covariances': [],
        'rmse': {},
        'compute_time': 0.0
    },
    'pf': {
        'estimates': np.zeros((N_steps, 13)),
        'rmse': {},
        'compute_time': 0.0
    }
}

ground_truth = {
    'states': states_100hz,
    'measurements_100hz': measurements_100hz,
    'measurements_10hz': measurements_10hz,
    'controls': controls_100hz,
    'time': time_100hz,
    'has_gps': has_gps_measurement,
    'has_imu': has_imu_measurement
}

## 6. Extended Kalman Filter (EKF)

Implement EKF using linearized dynamics for prediction and update steps.

### 6.1 Initialize

In [10]:
# Initialize EKF with perturbed initial state
np.random.seed(123)  # Different seed for initial perturbation
x_ekf_init = states_100hz[0] + np.random.multivariate_normal(np.zeros(13), P0)
# Normalize initial quaternion
x_ekf_init[6:10] = x_ekf_init[6:10] / np.linalg.norm(x_ekf_init[6:10])

print("EKF Initialized:")
print(f"  True initial position: {states_100hz[0, 0:3]}")
print(f"  EKF initial position:  {x_ekf_init[0:3]}")
print(f"  Position error: {np.linalg.norm(states_100hz[0, 0:3] - x_ekf_init[0:3]):.2f} m")

EKF Initialized:
  True initial position: [-14.02024609 -33.8160663    4.80178282]
  EKF initial position:  [-24.87655213 -23.84261183   7.6315678 ]
  Position error: 15.01 m


### 6.2 Prediction Step

In [11]:
def ekf_predict(x, P, u, dt, Q):
    """
    EKF prediction using linearized dynamics.
    
    Args:
        x: Current state estimate (13D)
        P: Current covariance (13x13)
        u: Control input (4D) [throttle, aileron, elevator, rudder]
        dt: Time step
        Q: Process noise covariance (13x13)
        
    Returns:
        x_pred: Predicted state
        P_pred: Predicted covariance
    """
    # YOUR CODE HERE
    # 1. Propagate state: x_pred = propagate_dynamics(x, u, dt)
    # 2. Compute state Jacobian F_c (continuous-time) using F(x, u) from dynamics.ipynb
    #    Convert to column vectors: x_vec, u_vec
    #    F_c = np.array(F(x_vec, u_vec))
    # 3. Discretize Jacobian: F_d = I + F_c * dt (first-order approximation)
    #    For better accuracy, use matrix exponential: F_d = expm(F_c * dt)
    # 4. Propagate covariance: P_pred = F_d @ P @ F_d.T + Q
    pass

### 6.3 Update Step

In [12]:
def ekf_update(x_pred, P_pred, z, R, measurement_model, measurement_jacobian):
    """
    EKF measurement update.
    
    Args:
        x_pred: Predicted state (13D)
        P_pred: Predicted covariance (13x13)
        z: Measurement (variable dimension: 4 for IMU, 6 for GPS)
        R: Measurement noise covariance (matches z dimension)
        measurement_model: Function to compute expected measurement h(x)
        measurement_jacobian: Function to compute Jacobian H(x)
        
    Returns:
        x_est: Updated state estimate
        P_est: Updated covariance
    """
    # YOUR CODE HERE
    # 1. Compute predicted measurement: z_pred = measurement_model(x_pred)
    # 2. Compute measurement Jacobian: H = measurement_jacobian(x_pred)
    # 3. Innovation: y = z - z_pred
    # 4. Innovation covariance: S = H @ P_pred @ H.T + R
    # 5. Kalman gain: K = P_pred @ H.T @ inv(S)
    # 6. Update state: x_est = x_pred + K @ y
    # 7. Update covariance: P_est = (I - K @ H) @ P_pred
    pass

### 6.4 Run EKF

In [13]:
# YOUR CODE HERE
# Loop through all time steps at 100 Hz:
# 1. Apply ekf_predict() with current control input
# 2. Always apply IMU update (100 Hz):
#      ekf_update(x_pred, P_pred, measurements_100hz[k], R_100hz, 
#                 measurement_model_100hz, measurement_jacobian_100hz)
# 3. If GPS available (has_gps_measurement[k] == True), also apply GPS update:
#      ekf_update(x_est, P_est, measurements_10hz[k], R_10hz,
#                 measurement_model_10hz, measurement_jacobian_10hz)
# 4. Store final estimate in results['ekf']['estimates'][k]
# 5. Optionally store covariance in results['ekf']['covariances']
# 6. Track total computation time
#
# Example structure:
# start_time = time.time()
# x_ekf = states_100hz[0] + np.random.multivariate_normal(np.zeros(13), P0)
# P_ekf = P0.copy()
# 
# for k in range(N_steps):
#     # Prediction
#     x_pred, P_pred = ekf_predict(x_ekf, P_ekf, controls_100hz[k], dt, Q_state)
#     
#     # IMU update (always available at 100 Hz)
#     x_ekf, P_ekf = ekf_update(x_pred, P_pred, measurements_100hz[k], R_100hz,
#                               measurement_model_100hz, measurement_jacobian_100hz)
#     
#     # GPS update (available at 10 Hz)
#     if has_gps_measurement[k]:
#         x_ekf, P_ekf = ekf_update(x_ekf, P_ekf, measurements_10hz[k], R_10hz,
#                                   measurement_model_10hz, measurement_jacobian_10hz)
#     
#     results['ekf']['estimates'][k] = x_ekf
# 
# results['ekf']['compute_time'] = time.time() - start_time

## 7. Unscented Kalman Filter (UKF)

Implement UKF using unscented transform for nonlinear propagation.

### 7.1 Sigma Point Generation

In [14]:
def generate_sigma_points(x, P, alpha, beta, kappa):
    """
    Generate sigma points for unscented transform.
    
    Args:
        x: State mean (13D)
        P: State covariance (13x13)
        alpha: Spread parameter (typically 1e-3)
        beta: Distribution parameter (2 for Gaussian)
        kappa: Secondary scaling (typically 0)
        
    Returns:
        sigma_points: Array of sigma points (27x13 for 13D state)
        weights_mean: Weights for mean computation (27,)
        weights_cov: Weights for covariance computation (27,)
    """
    # YOUR CODE HERE
    # 1. Compute lambda: lambda = alpha^2 * (n + kappa) - n
    # 2. Compute matrix square root: sqrt_P = cholesky((n + lambda) * P)
    # 3. Generate 2n+1 sigma points:
    #    sigma_0 = x
    #    sigma_i = x + sqrt_P[:, i-1] for i=1,...,n
    #    sigma_i = x - sqrt_P[:, i-n-1] for i=n+1,...,2n
    # 4. Compute weights:
    #    w_mean_0 = lambda / (n + lambda)
    #    w_cov_0 = lambda / (n + lambda) + (1 - alpha^2 + beta)
    #    w_i = 1 / (2 * (n + lambda)) for i=1,...,2n (both mean and cov)
    pass

### 7.2 Prediction

In [15]:
def ukf_predict(x, P, u, dt, Q, alpha, beta, kappa):
    """
    UKF prediction using unscented transform.
    
    Args:
        x: Current state estimate (13D)
        P: Current covariance (13x13)
        u: Control input (3D)
        dt: Time step
        Q: Process noise covariance (13x13)
        alpha, beta, kappa: UKF parameters
        
    Returns:
        x_pred: Predicted state mean
        P_pred: Predicted covariance
    """
    # YOUR CODE HERE
    # 1. Generate sigma points from current estimate
    # 2. Propagate each sigma point through dynamics using propagate_dynamics(sigma_i, u, dt)
    # 3. Compute predicted mean: x_pred = sum(w_i * sigma_i_pred)
    # 4. Compute predicted covariance: P_pred = sum(w_i * (sigma_i_pred - x_pred) @ (sigma_i_pred - x_pred).T) + Q
    pass

### 7.3 Update

In [16]:
def ukf_update(x_pred, P_pred, z, R, measurement_model, alpha, beta, kappa):
    """
    UKF measurement update using unscented transform.
    
    Args:
        x_pred: Predicted state (13D)
        P_pred: Predicted covariance (13x13)
        z: Measurement (variable dimension)
        R: Measurement noise covariance
        measurement_model: Measurement function h(x)
        alpha, beta, kappa: UKF parameters
        
    Returns:
        x_est: Updated state estimate
        P_est: Updated covariance
    """
    # YOUR CODE HERE
    # 1. Generate sigma points from predicted state
    # 2. Transform sigma points through measurement_model
    # 3. Compute predicted measurement mean: z_pred = sum(w_i * z_i)
    # 4. Compute innovation covariance: S = sum(w_i * (z_i - z_pred) @ (z_i - z_pred).T) + R
    # 5. Compute cross-covariance: P_xz = sum(w_i * (sigma_i - x_pred) @ (z_i - z_pred).T)
    # 6. Kalman gain: K = P_xz @ inv(S)
    # 7. Update state: x_est = x_pred + K @ (z - z_pred)
    # 8. Update covariance: P_est = P_pred - K @ S @ K.T
    pass

### 7.4 Run UKF

In [17]:
# YOUR CODE HERE
# Loop through all time steps:
# 1. Apply ukf_predict()
# 2. Apply IMU update with measurement_model_100hz
# 3. If GPS available, apply GPS update with measurement_model_10hz
# 4. Store results in results['ukf']
# 5. Track computation time
#
# Similar structure to EKF but using UKF functions

## 8. Particle Filter (PF)

Implement particle filter using Monte Carlo sampling.

### 8.1 Initialize Particles

In [18]:
def initialize_particles(x0, P0, N):
    """
    Generate initial particle cloud from Gaussian distribution.
    
    Args:
        x0: Initial state estimate (13D)
        P0: Initial covariance (13x13)
        N: Number of particles
        
    Returns:
        particles: (N x 13) array of particles
        weights: (N,) array of uniform weights
    """
    # YOUR CODE HERE
    # 1. Sample N particles from N(x0, P0)
    # 2. Initialize weights uniformly: weights = 1/N
    pass

### 8.2 Prediction (Propagate Particles)

In [19]:
def pf_predict(particles, u, dt, Q):
    """
    Propagate particles through dynamics with process noise.
    
    Args:
        particles: Current particles (N x 13)
        u: Control input (3D)
        dt: Time step
        Q: Process noise covariance (13x13)
        
    Returns:
        particles_pred: Propagated particles (N x 13)
    """
    # YOUR CODE HERE
    # For each particle i:
    #   1. Propagate through dynamics: x_i = propagate_dynamics(particles[i], u, dt)
    #   2. Add process noise: x_i += sample from N(0, Q)
    pass

### 8.3 Update (Compute Weights)

In [20]:
def pf_update_weights(particles, weights, z, R, measurement_model):
    """
    Update particle weights based on measurement likelihood.
    
    Args:
        particles: Particles (N x 13)
        weights: Current weights (N,)
        z: Measurement (variable dimension)
        R: Measurement noise covariance
        measurement_model: Measurement function h(x)
        
    Returns:
        weights_updated: Updated normalized weights (N,)
    """
    # YOUR CODE HERE
    # For each particle i:
    #   1. Compute expected measurement: z_i = measurement_model(particles[i])
    #   2. Compute likelihood: p(z | x_i) = exp(-0.5 * (z - z_i).T @ inv(R) @ (z - z_i))
    #      Or use multivariate_normal.pdf(z, mean=z_i, cov=R)
    #   3. Update weight: weights[i] *= likelihood
    # 4. Normalize weights: weights /= sum(weights)
    # 5. Handle numerical issues: if all weights near zero, reset to uniform
    pass

### 8.4 Resampling

In [21]:
def pf_resample(particles, weights):
    """
    Resample particles to avoid degeneracy.
    
    Args:
        particles: Particles (N x 13)
        weights: Normalized weights (N,)
        
    Returns:
        particles_resampled: Resampled particles (N x 13)
        weights_uniform: Uniform weights (N,) = 1/N
    """
    # YOUR CODE HERE
    # Implement systematic resampling:
    # 1. Compute cumulative sum of weights
    # 2. Generate N uniformly spaced samples in [0, 1]
    # 3. For each sample, find corresponding particle index
    # 4. Resample particles and reset weights to uniform
    # 
    # Alternatively, use multinomial resampling:
    # indices = np.random.choice(N, size=N, p=weights)
    # particles_resampled = particles[indices]
    pass

### 8.5 Run PF

In [22]:
# YOUR CODE HERE
# Initialize particles and weights
# Loop through all time steps:
# 1. Apply pf_predict()
# 2. Apply pf_update_weights() for IMU measurement (100 Hz)
# 3. If GPS available, apply pf_update_weights() for GPS measurement
# 4. Compute state estimate: x_est = sum(weights[i] * particles[i])
# 5. Check effective sample size: N_eff = 1 / sum(weights^2)
# 6. If N_eff < threshold (e.g., N/2), apply pf_resample()
# 7. Store estimate in results['pf']
# 8. Track computation time

## 9. Particle Filter Variants

Space for advanced particle filter implementations.

### Auxiliary Particle Filter

In [23]:
# Future implementation

### Regularized Particle Filter

In [24]:
# Future implementation

### Rao-Blackwellized Particle Filter

In [25]:
# Future implementation

## 10. Results Comparison

Visualize and analyze performance of all three filters.

### 10.1 Position Tracking

In [26]:
# YOUR CODE HERE
# Create subplots for px, py, pz
# Plot ground truth, EKF, UKF, PF estimates
# Include legend and labels

### 10.2 Velocity Tracking

In [27]:
# YOUR CODE HERE
# Create subplots for u, v, w (body frame velocities)
# Plot ground truth vs filter estimates

### 10.3 Attitude Tracking

In [28]:
# YOUR CODE HERE
# Convert quaternions to Euler angles (roll, pitch, yaw)
# Plot Euler angles for all filters vs ground truth
# Hint: Use rotation library or implement quaternion to Euler conversion

### 10.4 Angular Rate Tracking

In [29]:
# YOUR CODE HERE
# Create subplots for omega_x, omega_y, omega_z
# Plot ground truth vs filter estimates

### 10.5 RMSE Computation

In [30]:
# YOUR CODE HERE
# For each filter and each state component:
# Compute RMSE = sqrt(mean((estimate - truth)^2))
# Store in results[filter_name]['rmse']
# Create comparison table or bar chart

### 10.6 Computational Cost

In [31]:
# YOUR CODE HERE
# Create bar chart comparing computation times:
# - EKF: results['ekf']['compute_time']
# - UKF: results['ukf']['compute_time']
# - PF: results['pf']['compute_time']
# Also compute time per iteration for each filter

## 11. Analysis

Discussion and conclusions from the comparison study.

### Discussion Questions

1. **Accuracy**: Which filter achieved the lowest RMSE? Why might this be the case given the nonlinearity of the fixed-wing dynamics?

2. **Computational Cost**: How do the computation times compare? Is the increased accuracy of UKF/PF worth the computational overhead?

3. **Robustness**: How do the filters handle nonlinearity? Does the linearization in EKF cause significant errors?

4. **Practical Considerations**: 
   - When would you choose EKF over UKF/PF in a real UAV application?
   - How does measurement noise affect each filter?
   - What happens if process noise is increased?

5. **Extensions**: 
   - How might particle filter variants improve performance?
   - Could you combine strengths of different filters (e.g., EKF for linear parts, PF for highly nonlinear)?

In [32]:
# YOUR ANALYSIS HERE
# Provide written responses and supporting visualizations